# Component 3 - export for the web app (Stage A)

Converts the two trained networks to ONNX so the web ML service can run the validated pipeline **without PyTorch or
TensorFlow**, writes `cellular_config.json`, and checks that the web path gives the **same numbers as the validated
pipeline** on 13 validation images.

Output on Drive: `cellular_outputs/web_export/` -> download `models/cellular/` into `ml-service/models/cellular/`.
The `reference/` folder holds validation images and is **not** committed (patient data).

**GPU runtime** (the validated pipeline runs on the GPU for the comparison).


In [ ]:
# 1. Setup: mount Drive, switch the repo to the web-demo branch, copy images, set paths
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" checkout -- ml/cellular/notebooks/
!git -C "{REPO_DIR}" fetch origin
!git -C "{REPO_DIR}" checkout feature/web-demo-cellular
!git -C "{REPO_DIR}" pull --ff-only
!git -C "{REPO_DIR}" status -sb | head -1
%run "{REPO_DIR}/ml/cellular/colab_setup.py"


In [ ]:
# 2. Install StarDist, ONNX tools (tf2onnx without its old pins) + unit tests
!pip -q install tensorflow tf-keras stardist onnx onnxruntime
!pip -q install --no-deps tf2onnx
!python -m pytest tests/test_export_web.py -q


In [ ]:
# 3. Export + parity check (GPU, ~5 min)
!python -m export_web 2>&1 | grep -v -E "Warning|warn|oneDNN|absl"


In [ ]:
# 4. What to download
import os, json
d = f"{OUTPUT_ROOT}/web_export/models/cellular"
for f in sorted(os.listdir(d)):
    print(f"{f:28s} {os.path.getsize(os.path.join(d, f)) / 1e6:8.2f} MB")
c = json.load(open(f"{d}/cellular_config.json"))
print(json.dumps({k: c[k] for k in ("source", "models")}, indent=2)[:3000])
